In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.models import Model


# Force TensorFlow to use CPU if needed
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# Set up InceptionV3 as feature extractor
base_model = InceptionV3(weights="imagenet", include_top=False, input_shape=(90, 75, 3))
x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
feature_extractor = Model(inputs=base_model.input, outputs=x)
print("Feature extractor model loaded successfully")

def extract_features(image):
    """Extract features using InceptionV3 feature extractor"""
    image = np.expand_dims(image, axis=0)  # Add batch dimension
    image = image / 255.0  # Normalize
    features = feature_extractor.predict(image, verbose=0)
    return features.flatten()

# Import MediaPipe components for hand detection
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=2,min_detection_confidence=0.5, min_tracking_confidence=0.5)
mp_draw = mp.solutions.drawing_utils

def extract_hand_roi(frame):
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(img_rgb)
    
    if not result.multi_hand_landmarks:
        return frame, None
    
    # Track coordinates for all detected hands
    all_x_min, all_y_min = float('inf'), float('inf')
    all_x_max, all_y_max = 0, 0
    
    # Check if hands were detected
    hand_detected = False
    
    custom_landmark_drawing_spec = mp_draw.DrawingSpec(
        color=(0, 255, 0),  # Green color for landmarks
        thickness=1,        # Thinner lines (default is 2)
        circle_radius=1     # Smaller circles (default is 2)
    )
    
    custom_connection_drawing_spec = mp_draw.DrawingSpec(
        color=(0, 0, 255),  # Red color for connections
        thickness=1,        # Thinner lines (default is 2)
        circle_radius=1     # Smaller circles at joints (default is 2)
    )
    
    for hand_landmarks in result.multi_hand_landmarks:
        # Draw hand landmarks
        mp_draw.draw_landmarks(
            frame, 
            hand_landmarks, 
            mp_hands.HAND_CONNECTIONS,
            landmark_drawing_spec=custom_landmark_drawing_spec,
            connection_drawing_spec=custom_connection_drawing_spec
        )
        
        # Find boundaries for this hand
        x_min = min([lm.x for lm in hand_landmarks.landmark])
        y_min = min([lm.y for lm in hand_landmarks.landmark])
        x_max = max([lm.x for lm in hand_landmarks.landmark])
        y_max = max([lm.y for lm in hand_landmarks.landmark])
        
        # Update the overall boundaries to encompass all hands
        all_x_min = min(all_x_min, x_min)
        all_y_min = min(all_y_min, y_min)
        all_x_max = max(all_x_max, x_max)
        all_y_max = max(all_y_max, y_max)
        
        hand_detected = True
    
    if hand_detected:
        h, w, _ = frame.shape
        all_x_min, all_x_max = max(0, int(all_x_min * w)), min(w, int(all_x_max * w))
        all_y_min, all_y_max = max(0, int(all_y_min * h)), min(h, int(all_y_max * h))
        
        # Add padding around the hands
        pad_x = int((all_x_max - all_x_min) * 0.2)
        pad_y = int((all_y_max - all_y_min) * 0.2)
        
        all_x_min = max(0, all_x_min - pad_x)
        all_y_min = max(0, all_y_min - pad_y)
        all_x_max = min(w, all_x_max + pad_x)
        all_y_max = min(h, all_y_max + pad_y)
        
        # Draw bounding box around all detected hands
        # cv2.rectangle(frame, (all_x_min, all_y_min), (all_x_max, all_y_max), (0, 255, 0), 2)
        
        # Extract region containing all hands
        roi = frame[all_y_min:all_y_max, all_x_min:all_x_max]
        if roi.size > 0:  # Check if ROI is valid
            # Resize to specific dimensions required by feature extractor
            roi_resized = cv2.resize(roi, (75, 90))
            return frame, roi_resized
    
    return frame, None

def process_images_in_folder(folder_path):
    # Get list of image files in the folder
    valid_extensions = ['.jpg', '.jpeg', '.png', '.bmp']
    image_files = [f for f in os.listdir(folder_path) if os.path.splitext(f.lower())[1] in valid_extensions]
    
    if not image_files:
        print(f"No valid image files found in {folder_path}")
        return
    
    for image_file in image_files:
        image_path = os.path.join(folder_path, image_file)
        print(f"\nProcessing: {image_file}")
        
        # Read the image
        image = cv2.imread(image_path)
        if image is None:
            print(f"Failed to load image: {image_path}")
            continue
        
        # Extract ROI using MediaPipe hand detection
        processed_image, hand_roi = extract_hand_roi(image.copy())
        
        try:
            # If no hands detected, use the full image resized to required dimensions
            if hand_roi is None:
                print("No hands detected, using full image for feature extraction")
                # Resize the full image to the required dimensions
                hand_roi = cv2.resize(image, (75, 90))
            
            # Extract features
            features = extract_features(hand_roi)
            
            # Display the original image, ROI, and feature visualization
            plt.figure(figsize=(15, 8))
            
            # Display processed image
            plt.subplot(2, 2, 1)
            plt.title("Processed Image")
            processed_rgb = cv2.cvtColor(processed_image, cv2.COLOR_BGR2RGB)
            plt.imshow(processed_rgb)
            plt.axis('off')
            
            # Display ROI (either hand ROI or resized full image)
            plt.subplot(2, 2, 2)
            roi_title = "Hand ROI (75x90)" if hand_roi is not None else "Full Image Resized (75x90)"
            plt.title(roi_title)
            roi_rgb = cv2.cvtColor(hand_roi, cv2.COLOR_BGR2RGB)
            plt.imshow(roi_rgb)
            plt.axis('off')
            
            # Display feature visualization
            plt.subplot(2, 2, 3)
            plt.title(f"Feature Vector (Length: {len(features)})")
            plt.plot(features)
            plt.grid(True)
            
            # Display feature heatmap
            plt.subplot(2, 2, 4)
            plt.title("Feature Heatmap")
            # Reshape features to make a more visual heatmap
            feature_map_size = int(np.sqrt(len(features)))
            reshaped_features = features[:feature_map_size**2].reshape(feature_map_size, feature_map_size)
            plt.imshow(reshaped_features, cmap='viridis')
            plt.colorbar(label="Feature Value")
            
            plt.tight_layout()
            plt.suptitle(f"Feature Extraction Results for {image_file}", y=1.02)
            plt.subplots_adjust(top=0.9)
            
            # Print feature statistics
            print(f"Feature vector shape: {features.shape}")
            print(f"Feature min: {features.min():.4f}, max: {features.max():.4f}")
            print(f"Feature mean: {features.mean():.4f}, std: {features.std():.4f}")
            
            plt.show()
            
        except Exception as e:
            print(f"Error during feature extraction: {e}")
            # traceback.print_exc()  # Print the complete traceback for better debugging
            continue

if __name__ == "__main__":
    folder_path = "/mnt/MainDrive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/archive_2/Test/2"
    if os.path.isdir(folder_path):
        process_images_in_folder(folder_path)
    else:
        print("Invalid folder path.")

In [2]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.models import Model
import time
import ctypes
ctypes.CDLL("/usr/lib/libcudnn.so.9.7.0")


# Set up InceptionV3 as feature extractor
base_model = InceptionV3(weights="imagenet", include_top=False, input_shape=(90, 75, 3))
x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
feature_extractor = Model(inputs=base_model.input, outputs=x)
print("Feature extractor model loaded successfully")

def extract_features(image):
    """Extract features using InceptionV3 feature extractor"""
    if image.shape[0] != 90 or image.shape[1] != 75:
        print(f"Resizing image from {image.shape} to (90, 75, 3)")
        image = cv2.resize(image, (75, 90))
        
    image = np.expand_dims(image, axis=0)  # Add batch dimension
    image = image / 255.0  # Normalize
    features = feature_extractor.predict(image, verbose=0)
    return features.flatten()

# Import MediaPipe components for hand detection
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=2, min_detection_confidence=0.5, min_tracking_confidence=0.5)
mp_draw = mp.solutions.drawing_utils

def extract_hand_roi(frame):
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(img_rgb)
    
    if not result.multi_hand_landmarks:
        return frame, None, 0
    
    # Track coordinates for all detected hands
    all_x_min, all_y_min = float('inf'), float('inf')
    all_x_max, all_y_max = 0, 0
    
    # Check if hands were detected
    hand_detected = False
    num_hands = 0
    
    # Create custom drawing specs for smaller landmarks
    custom_landmark_drawing_spec = mp_draw.DrawingSpec(
        color=(0, 255, 0),  # Green color for landmarks
        thickness=1,        # Thinner lines
        circle_radius=1     # Smaller circles
    )
    
    custom_connection_drawing_spec = mp_draw.DrawingSpec(
        color=(255, 0, 0),  # Red color for connections
        thickness=1,        # Thinner lines
        circle_radius=1     # Smaller circles at joints
    )
    
    for hand_landmarks in result.multi_hand_landmarks:
        num_hands += 1
        # Draw hand landmarks
        mp_draw.draw_landmarks(
            frame, 
            hand_landmarks, 
            mp_hands.HAND_CONNECTIONS,
            landmark_drawing_spec=custom_landmark_drawing_spec,
            connection_drawing_spec=custom_connection_drawing_spec
        )
        
        # Find boundaries for this hand
        x_min = min([lm.x for lm in hand_landmarks.landmark])
        y_min = min([lm.y for lm in hand_landmarks.landmark])
        x_max = max([lm.x for lm in hand_landmarks.landmark])
        y_max = max([lm.y for lm in hand_landmarks.landmark])
        
        # Update the overall boundaries to encompass all hands
        all_x_min = min(all_x_min, x_min)
        all_y_min = min(all_y_min, y_min)
        all_x_max = max(all_x_max, x_max)
        all_y_max = max(all_y_max, y_max)
        
        hand_detected = True
    
    if hand_detected:
        h, w, _ = frame.shape
        all_x_min, all_x_max = max(0, int(all_x_min * w)), min(w, int(all_x_max * w))
        all_y_min, all_y_max = max(0, int(all_y_min * h)), min(h, int(all_y_max * h))
        
        # Add padding around the hands
        pad_x = int((all_x_max - all_x_min) * 0.2)
        pad_y = int((all_y_max - all_y_min) * 0.2)
        
        all_x_min = max(0, all_x_min - pad_x)
        all_y_min = max(0, all_y_min - pad_y)
        all_x_max = min(w, all_x_max + pad_x)
        all_y_max = min(h, all_y_max + pad_y)
        
        # Draw bounding box around all detected hands
        cv2.rectangle(frame, (all_x_min, all_y_min), (all_x_max, all_y_max), (0, 255, 0), 2)
        
        # Extract region containing all hands
        roi = frame[all_y_min:all_y_max, all_x_min:all_x_max]
        if roi.size > 0:  # Check if ROI is valid
            # Resize to specific dimensions required by feature extractor
            roi_resized = cv2.resize(roi, (75, 90))
            return frame, roi_resized, num_hands
    
    return frame, None, num_hands

def main():
    # Open webcam
    cap = cv2.VideoCapture(1)  # Use 0 for the default camera, or try 1, 2, etc. for other cameras
    
    # Check if webcam is opened correctly
    if not cap.isOpened():
        print("Error: Could not open webcam")
        return
    
    # Initialize feature visualization arrays
    feature_history = []
    max_history = 30  # Number of frames to keep in history
    latest_features = None
    
    # For FPS calculation
    prev_time = 0
    curr_time = 0
    
    print("Starting real-time hand feature extraction...")
    print("Press 'q' to quit")
    
    while True:
        # Capture frame
        ret, frame = cap.read()
        if not ret:
            print("Error: Can't receive frame")
            break
        
        # Mirror the frame horizontally for a more natural view
        frame = cv2.flip(frame, 1)
        
        # Calculate FPS
        curr_time = time.time()
        fps = 1 / (curr_time - prev_time) if prev_time > 0 else 0
        prev_time = curr_time
        
        # Extract hand ROI and process features
        processed_frame, hand_roi, num_hands = extract_hand_roi(frame)
        
        # Create a display window for all visualizations
        h, w, _ = frame.shape
        display = np.zeros((h, w + 400, 3), dtype=np.uint8)
        
        # Add the original frame with hand detection to the display
        display[:h, :w] = processed_frame
        
        # Extract features if hand is detected
        if hand_roi is not None:
            try:
                # Extract features
                features = extract_features(hand_roi)
                latest_features = features
                
                # Add to history (for visualization)
                feature_history.append(features)
                if len(feature_history) > max_history:
                    feature_history.pop(0)
                
                # Display ROI in the right panel
                roi_h, roi_w, _ = hand_roi.shape
                roi_scale = 2
                scaled_roi = cv2.resize(hand_roi, (roi_w * roi_scale, roi_h * roi_scale))
                s_h, s_w, _ = scaled_roi.shape
                
                # Position the ROI in the right panel
                start_x = w + 50
                start_y = 50
                display[start_y:start_y + s_h, start_x:start_x + s_w] = scaled_roi
                
                # Add a label
                cv2.putText(display, "Hand ROI", (start_x, start_y - 10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                
                # Draw feature visualization
                if latest_features is not None:
                    # 1. Plot first few feature values as a bar graph
                    num_features_to_show = min(20, len(latest_features))
                    feature_values = latest_features[:num_features_to_show]
                    max_value = max(1.0, np.max(np.abs(feature_values))) # For normalization
                    
                    bar_start_x = w + 50
                    bar_start_y = start_y + s_h + 50
                    bar_width = 15
                    bar_height_max = 100
                    
                    # Draw feature bars
                    for i, val in enumerate(feature_values):
                        normalized_val = val / max_value
                        bar_height = int(abs(normalized_val) * bar_height_max)
                        bar_x = bar_start_x + i * bar_width
                        
                        # Color based on positive/negative value
                        color = (0, 255, 0) if val >= 0 else (0, 0, 255)
                        
                        # Draw bar (going up for positive, down for negative)
                        if val >= 0:
                            cv2.rectangle(display, 
                                         (bar_x, bar_start_y), 
                                         (bar_x + bar_width - 2, bar_start_y - bar_height), 
                                         color, -1)
                        else:
                            cv2.rectangle(display, 
                                         (bar_x, bar_start_y), 
                                         (bar_x + bar_width - 2, bar_start_y + bar_height), 
                                         color, -1)
                    
                    # Draw feature heatmap
                    heatmap_start_y = bar_start_y + 150
                    feature_map_size = int(np.sqrt(len(features)))
                    reshaped_features = features[:feature_map_size**2].reshape(feature_map_size, feature_map_size)
                    
                    # Normalize the features to 0-255 for visualization
                    norm_features = cv2.normalize(reshaped_features, None, 0, 255, cv2.NORM_MINMAX)
                    norm_features = norm_features.astype(np.uint8)
                    
                    # Apply a colormap
                    heatmap = cv2.applyColorMap(norm_features, cv2.COLORMAP_JET)
                    
                    # Resize for better visibility
                    heatmap_size = 150
                    heatmap_resized = cv2.resize(heatmap, (heatmap_size, heatmap_size))
                    
                    # Add to display
                    display[heatmap_start_y:heatmap_start_y + heatmap_size, 
                           w + 125:w + 125 + heatmap_size] = heatmap_resized
                    
                    # Add label for heatmap
                    cv2.putText(display, "Feature Heatmap", (w + 125, heatmap_start_y - 10),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
                    
            except Exception as e:
                print(f"Error processing features: {e}")
        
        # Add FPS counter
        cv2.putText(display, f"FPS: {fps:.1f}", (10, 30), 
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        # Add number of hands detected
        cv2.putText(display, f"Hands: {num_hands}", (10, 70), 
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        # Display
        cv2.imshow('Real-time Hand Feature Extraction', display)
        
        # Exit on 'q' key press
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    # Release resources
    cap.release()
    cv2.destroyAllWindows()
    print("Application closed")

if __name__ == "__main__":
    main()

Feature extractor model loaded successfully
Starting real-time hand feature extraction...
Press 'q' to quit


I0000 00:00:1742155743.115982  583772 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1742155743.117435  618902 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 24.3.4-arch1.1), renderer: AMD Radeon Graphics (radeonsi, renoir, LLVM 19.1.7, DRM 3.54, 6.6.72-1-lts)
W0000 00:00:1742155743.140753  618886 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1742155743.163031  618883 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Error processing features: could not broadcast input array from shape (150,150,3) into shape (50,150,3)
Error processing features: could not broadcast input array from shape (150,150,3) into shape (50,150,3)
Error processing features: could not broadcast input array from shape (150,150,3) into shape (50,150,3)
Error processing features: could not broadcast input array from shape (150,150,3) into shape (50,150,3)
Error processing features: could not broadcast input array from shape (150,150,3) into shape (50,150,3)
Error processing features: could not broadcast input array from shape (150,150,3) into shape (50,150,3)
Error processing features: could not broadcast input array from shape (150,150,3) into shape (50,150,3)
Error processing features: could not broadcast input array from shape (150,150,3) into shape (50,150,3)
Error processing features: could not broadcast input array from shape (150,150,3) into shape (50,150,3)
Error processing features: could not broadcast input array from 

In [2]:
import cv2
import numpy as np
import mediapipe as mp
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Model


2025-03-17 01:15:02.361124: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742154302.376035  568580 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742154302.380712  568580 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-17 01:15:02.396321: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
base_model = InceptionV3(weights="imagenet", include_top=False, input_shape=(50, 60, 3))
x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
feature_extractor = Model(inputs=base_model.input, outputs=x)

def extract_features(image):
    image = np.expand_dims(image, axis=0)  # Add batch dimension
    image = image / 255.0  # Normalize
    features = feature_extractor.predict(image)
    return features.flatten()

mp_hands = mp.solutions.hands
# Changed to detect up to 2 hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=2)
mp_draw = mp.solutions.drawing_utils

def extract_hand_roi(frame):
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(img_rgb)
    
    if not result.multi_hand_landmarks:
        return frame, None
    
    # Track coordinates for all detected hands
    all_x_min, all_y_min = float('inf'), float('inf')
    all_x_max, all_y_max = 0, 0
    
    # Check if hands were detected
    hand_detected = False
    
    for hand_landmarks in result.multi_hand_landmarks:
        # Draw hand landmarks
        mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
        
        # Find boundaries for this hand
        x_min = min([lm.x for lm in hand_landmarks.landmark])
        y_min = min([lm.y for lm in hand_landmarks.landmark])
        x_max = max([lm.x for lm in hand_landmarks.landmark])
        y_max = max([lm.y for lm in hand_landmarks.landmark])
        
        # Update the overall boundaries to encompass all hands
        all_x_min = min(all_x_min, x_min)
        all_y_min = min(all_y_min, y_min)
        all_x_max = max(all_x_max, x_max)
        all_y_max = max(all_y_max, y_max)
        
        hand_detected = True
    
    if hand_detected:
        h, w, _ = frame.shape
        all_x_min, all_x_max = max(0, int(all_x_min * w)), min(w, int(all_x_max * w))
        all_y_min, all_y_max = max(0, int(all_y_min * h)), min(h, int(all_y_max * h))
        
        # Add padding around the hands
        pad_x = int((all_x_max - all_x_min) * 0.2)
        pad_y = int((all_y_max - all_y_min) * 0.2)
        
        all_x_min = max(0, all_x_min - pad_x)
        all_y_min = max(0, all_y_min - pad_y)
        all_x_max = min(w, all_x_max + pad_x)
        all_y_max = min(h, all_y_max + pad_y)
        
        # Draw bounding box around all detected hands
        # cv2.rectangle(frame, (all_x_min, all_y_min), (all_x_max, all_y_max), (0, 255, 0), 2)
        
        # Extract region containing all hands
        roi = frame[all_y_min:all_y_max, all_x_min:all_x_max]
        if roi.size > 0:  # Check if ROI is valid
            # Maintain aspect ratio while resizing
            roi_h, roi_w = all_y_max - all_y_min, all_x_max - all_x_min
            
            # Set a target width and calculate height proportionally
            # target_width = 100
            # target_height = int(roi_h * (target_width / roi_w))
            target_width = 50
            target_height =60
            
            roi_resized = cv2.resize(roi, (target_width, target_height))
            return frame, roi_resized
    
    return frame, None

def main():
    cap = cv2.VideoCapture(1)
    
    # Check if webcam is opened correctly
    if not cap.isOpened():
        print("Error: Could not open webcam")
        return
    
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Error: Can't receive frame")
            break
        
        # Mirror the frame horizontally for a more natural view
        frame = cv2.flip(frame, 1)
        
        # Extract hand ROI
        processed_frame, hand_roi = extract_hand_roi(frame)
        
        # Create a display window
        h, w, _ = frame.shape
        display = np.zeros((h, w + 150, 3), dtype=np.uint8)
        
        # Add the original frame to the display
        display[:h, :w] = processed_frame
        
        # Add the hand ROI if available
        if hand_roi is not None:
            # Calculate position for the ROI (centered in the right panel)
            roi_h, roi_w, _ = hand_roi.shape
            start_x = w + (150 - roi_w) // 2
            start_y = (h - roi_h) // 2
            display[start_y:start_y + roi_h, start_x:start_x + roi_w] = hand_roi
            
            # Add a label
            cv2.putText(display, "Hands ROI", (w + 30, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            
            # Show number of hands detected (FIXED)
            num_hands = 0
            result = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if result.multi_hand_landmarks:
                num_hands = len(result.multi_hand_landmarks)
            cv2.putText(display, f"{num_hands} hand(s)", (w + 30, 60), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        cv2.imshow('Hand ROI Extractor', display)
        
        # Quit on 'q' key press
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

ValueError: Input size must be at least 75x75; Received: input_shape=(50, 60, 3)

In [1]:
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, TimeDistributed
import mediapipe as mp
import numpy as np
import cv2
import os
from sklearn.model_selection import train_test_split


2025-03-09 04:31:53.890531: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1741474913.948176  393635 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1741474913.965840  393635 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-09 04:31:54.094055: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
SEQ_LENGTH = 10
IMG_SIZE = (50, 60)  # Matches original architecture
BATCH_SIZE = 8
EPOCHS = 20

# Load data (assuming data is already preprocessed and stored in numpy arrays)
data = np.load('data.npy')
labels = np.load('labels.npy')

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, random_state=42)

# Define the model architecture
def create_model():
    base_model = InceptionV3(include_top=False, weights='imagenet', input_shape=(IMG_SIZE[1], IMG_SIZE[0], 3))
    base_model.trainable = False
    
    model = Sequential()
    model.add(TimeDistributed(base_model, input_shape=(SEQ_LENGTH, IMG_SIZE[1], IMG_SIZE[0], 3)))
    model.add(TimeDistributed(Dense(1024, activation='relu')))
    model.add(LSTM(512, return_sequences=False, dropout=0.5))
    model.add(Dense(256, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(5, activation='softmax'))  # Assuming 5 gesture classes
    
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Create the model
model = create_model()

# Train the model
history = model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_data=(X_test, y_test))

# Save the model
model.save('gesture_model.h5')
